<a href="https://colab.research.google.com/github/OswaldoVelezDev/DB_BIG_DATA/blob/main/BD_Analitica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Instalación de Duck DB**

In [ ]:
# Instalacion de Duck DB
!pip install duckdb

# Importamos
import duckdb
import pandas as pd

# Verificar instalación
print(duckdb.__version__)

1.3.2


In [ ]:
# Cargamos el drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Usamos la ruta del archivo CSV que ya definimos
csv_file_path = '/content/drive/MyDrive/Ingeniería/Big Data/total_incidentes_transito.csv'

try:
  df = pd.read_csv(csv_file_path, encoding='latin-1')
  print('Primeras 5 filas del archivo CSV:')
  display(df.head())
except Exception as e:
  print(f"Error al leer el archivo CSV: {e}")

Primeras 5 filas del archivo CSV:


,OBJECTID,Shape,radicado,fecha,hora,dia,periodo,clase,direccion,direccion_enc,...,barrio,comuna,diseno,dia_nombre,mes,mes_nombre,longitud,latitud,x_origen_nacional,y_origen_nacional
0,1,"(4713547.5105, 2247612.977400001)",1580885,2017-05-05 00:00:00,02:00 PM,5,2017,Choque,CL 32 Norte CR 69,CL 032 069 000 00000,...,Rosales,Belén,Tramo de via,VIERNES,5,NaN,-75.589659,6.234581,4.713548e+06,2.247613e+06
1,2,"(4715357.8105, 2243807.580100002)",1585081,2017-06-06 00:00:00,11:20 AM,6,2017,Choque,CL 4 Sur CR 43 B,CL S 004 043 B 000 00000,...,Patio Bonito,El Poblado,Tramo de via,MARTES,6,NaN,-75.573137,6.200258,4.715358e+06,2.243808e+06
2,3,"(4710262.4967, 2249847.019700002)",1581508,2017-05-10 00:00:00,12:00 PM,10,2017,Choque,CL 40 CR 105,CL 040 105 000 00000,...,San Javier No.1,San Javier,Tramo de via,MIÉRCOLES,5,NaN,-75.619437,6.254631,4.710262e+06,2.249847e+06
3,4,"(4715512.4542000005, 2246920.8745000027)",1581862,2017-05-12 00:00:00,05:30 PM,12,2017,Caida Ocupante,CL 28 CR 44,CL 028 044 000 00000,...,Barrio Colombia,El Poblado,Tramo de via,VIERNES,5,NaN,-75.571877,6.228411,4.715512e+06,2.246921e+06
4,5,"(4713597.201000002, 2252326.8505000006)",1578199,2017-04-14 00:00:00,04:30 AM,14,2017,Choque,CL 76 CR 80,CL 076 080 000 00000,...,Villa Flora,Robledo,Tramo de via,VIERNES,4,NaN,-75.589420,6.277200,4.713597e+06,2.252327e+06


### **Creación del esquema en Duck DB**

In [ ]:
conn = duckdb.connect('accidentes_medellin.db')

# Creamos la dimension de tiempo con columnas: fecha, hora, periodo,
# mes y dia_nombre
conn.execute(f'''
  CREATE TABLE IF NOT EXISTS dim_tiempo AS
  SELECT DISTINCT
    CAST (strftime(CAST(fecha AS DATE), '%Y%m%d') AS INTEGER) AS id_fecha,
    CAST(fecha AS DATE) AS fecha_completa,
    periodo AS anio,
    mes AS mes,
    mes_nombre AS mes_nombre,
    dia_nombre AS dia_nombre,
    CASE
      WHEN CAST(split_part(hora, ':', 1) AS INT) BETWEEN 0 AND 5 THEN 'Madrugada'
      WHEN CAST(split_part(hora, ':', 1) AS INT) BETWEEN 6 AND 11 THEN 'Mañana'
      WHEN CAST(split_part(hora, ':', 1) AS INT) BETWEEN 12 AND 17 THEN 'Tarde'
      ELSE 'Noche'
    END AS franja_horaria
  FROM read_csv_auto('{csv_file_path}', delim = ',',
                      header = true, encoding ='latin-1', strict_mode=false)
  WHERE fecha IS NOT NULL
''')

In [ ]:
# Creamos la dimension de ubicacion con columnas: barrio y comuna
conn.execute(f'''
  CREATE TABLE IF NOT EXISTS dim_ubicacion AS
  SELECT DISTINCT
    ROW_NUMBER() OVER () AS id_ubicacion,
    barrio, comuna
  FROM read_csv_auto('{csv_file_path}', delim = ',',
                    header = true, encoding ='latin-1', strict_mode=false)
  WHERE barrio IS NOT NULL
''')

In [ ]:
# Creamos la dimesion de clase con columna: clase
conn.execute(f'''
  CREATE TABLE IF NOT EXISTS dim_clase AS
  SELECT DISTINCT
    ROW_NUMBER() OVER () AS id_clase,
    TRIM(clase) AS clase_accidente
  FROM read_csv_auto('{csv_file_path}', delim = ',',
                    header = true, encoding ='latin-1', strict_mode=false)
  WHERE clase IS NOT NULL
''')

In [ ]:
# Creamos la dimension de gravedad con columna: gravedad
conn.execute(f'''
  CREATE TABLE IF NOT EXISTS dim_gravedad AS
  SELECT DISTINCT
    ROW_NUMBER() OVER () AS id_gravedad,
    gravedad AS nivel_gravedad,
    CASE gravedad
      WHEN 'SOLO DAÑOS' THEN 1
      WHEN 'HERIDO' THEN 2
      WHEN 'MUERTO' THEN 3
      ELSE 0
    END AS orden_severidad
  FROM read_csv_auto('{csv_file_path}', delim = ',',
                    header = true, encoding ='latin-1',  strict_mode=false)
  WHERE gravedad IS NOT NULL
''')

In [ ]:
# Creamos la dimension de via con columna: diseño
conn.execute(f'''
  CREATE TABLE IF NOT EXISTS dim_via AS
  SELECT DISTINCT
    ROW_NUMBER() OVER () AS id_via,
    diseno AS tipo_via
  FROM read_csv_auto('{csv_file_path}', delim = ',',
                    header = true, encoding ='latin-1',  strict_mode=false)
  WHERE diseno IS NOT NULL
''')

### **Carga e ingesta de datos - Tabla de hecho**

In [ ]:
# -- fact_accidentes: columnas: OBJECTID, fecha, barrio, comuna,
#                     clase, gravedad, diseno, latitud, longitud
conn.execute(f'''
  CREATE TABLE IF NOT EXISTS fact_accidentes AS
  SELECT
    src.OBJECTID AS id_accidente,
    CAST(strftime(CAST(src.fecha AS DATE),'%Y%m%d') AS INTEGER) AS id_fecha,
    u.id_ubicacion,
    c.id_clase,
    g.id_gravedad,
    v.id_via,
    src.latitud,
    src.longitud
  FROM read_csv_auto('{csv_file_path}',
                     delim=',', header=true, encoding='latin-1', strict_mode=false, ignore_errors=true) src
  JOIN dim_ubicacion u ON u.barrio = src.barrio
                     AND u.comuna = src.comuna
  JOIN dim_clase     c ON c.clase_accidente = TRIM(src.clase)
  JOIN dim_gravedad  g ON g.nivel_gravedad  = src.gravedad
  JOIN dim_via       v ON v.tipo_via        = src.diseno
  WHERE src.fecha IS NOT NULL
''')
print('Registros cargados:', conn.execute('SELECT COUNT(*) FROM fact_accidentes').fetchone()[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))